In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install langgraph
%pip install langchain-groq
%pip install langchain-core
%pip install langchain

In [0]:
%pip install -U mlflow langchain-core

In [0]:
%restart_python

In [0]:
import sys
import os
import mlflow
from langchain_core.messages import HumanMessage

# Ensure the root directory is in the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import the fully compiled graph
from src.agent.graph import compliance_app

print("===================================================")
print("🚀 INITIATING E2E PIPELINE TEST")
print("===================================================\n")

# 1. Define a realistic user query
test_query = "Can you run a full compliance audit on account C1231006815? I need to know if there are any discrepancies or high-risk flags."

initial_state = {
    "messages": [HumanMessage(content=test_query)],
    "next_node": "",
    "is_security_breach": False
}

print(f"👤 USER PROMPT: {test_query}\n")
print("--- Execution Trace ---")

# 2. Start the MLflow Run
with mlflow.start_run(run_name="E2E_Compliance_Audit"):
    
    # 3. Stream the execution of the graph
    # Using .stream() instead of .invoke() allows us to print the output of each node as it finishes
    for output in compliance_app.stream(initial_state):
        
        # Identify which node just ran
        for node_name, state_update in output.items():
            print(f"✅ Node Completed: [{node_name}]")
            
            # If the node generated a message, print a snippet of what it did
            if "messages" in state_update and state_update["messages"]:
                latest_msg = state_update["messages"][-1]
                
                # Check if it was a tool call
                if hasattr(latest_msg, 'tool_calls') and latest_msg.tool_calls:
                    print(f"   ↳ Action: Requested Tool -> {latest_msg.tool_calls[0]['name']}")
                else:
                    # Print the first 100 characters of the text generated
                    preview = latest_msg.content[:100].replace('\n', ' ') + "..."
                    print(f"   ↳ Output: {preview}")
            print("-" * 23)

    print("\n===================================================")
    print("🏁 FINAL COMPLIANCE REPORT (EGRESS SCRUBBED)")
    print("===================================================\n")
    
    # Fetch the final message from the completed state
    final_state = compliance_app.invoke(initial_state)
    print(final_state["messages"][-1].content)
    
    print("\n[MLflow Tracking Active: Check your Databricks Experiments tab to view the full execution graph]")